# Gradient Descent, SGD, and Momentum

The normal equations give least squares in one exact solve, but forming and
solving that system can be expensive when the data have many rows or features.
Iterative methods replace one expensive solve by a sequence of cheaper updates.
The central questions are then: how large should each update be, how noisy can
the gradient be, and how does the geometry of the loss affect convergence?

Least squares is a useful laboratory because we know its exact optimum and can
check every optimization claim. We will:

1. compare the computational cost of an exact solve, a full gradient, and a
   minibatch gradient;
2. isolate the role of the learning rate on a one-dimensional quadratic;
3. run gradient descent on a two-dimensional least-squares objective;
4. trade cheaper updates for noisier minibatch gradients;
5. use momentum to accelerate an ill-conditioned problem; and
6. modify the update for the nondifferentiable lasso objective.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

SEED = 26510
rng = np.random.default_rng(SEED)
_ = torch.manual_seed(SEED)

## A controlled least-squares problem

Real data mix together optimization geometry, noise, and modeling choices. A
simulation lets us control the geometry while keeping the optimum known. The
next cell creates a two-feature design with population condition number
$\kappa=8$: its eigenvalues run from 1 to $1/\kappa$, then the axes are rotated
so that neither coordinate alone is the slow direction.

The printed sample eigenvalues show the curvature available in the steep and
flat directions. Their ratio is the realized condition number; the largest
eigenvalue also determines the stable learning-rate bound used below. Because
the Hessian `H` is square and full rank, `np.linalg.solve(H, right_hand_side)`
computes the exact benchmark. NumPy is solving the linear system we derived; it
is not choosing the regression model for us.

In [ ]:
def simulate(n, d, kappa, noise=.5, seed=0):
    gen = np.random.default_rng(seed)
    eigs = np.logspace(0, -np.log10(kappa), d)
    angle = np.pi/4
    rot = (np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
           if d == 2 else np.linalg.qr(gen.standard_normal((d, d)))[0])
    root = rot @ np.diag(np.sqrt(eigs)) @ rot.T
    Xs = gen.standard_normal((n, d)) @ root
    theta = np.array([1.4, -0.9]) if d == 2 else gen.standard_normal(d)
    return Xs, Xs@theta + noise*gen.standard_normal(n)

X, y = simulate(2000, 2, 8.0, seed=SEED)
H = X.T@X/len(X)
right_hand_side = X.T@y/len(X)
theta_star = np.linalg.solve(H, right_hand_side)
print('realized condition number:', round(np.linalg.cond(H), 2))
print('Hessian eigenvalues:', np.linalg.eigvalsh(H).round(4))
print('largest stable step 2/lambda_max:', round(2/np.linalg.eigvalsh(H).max(), 3))

## 1. Why use an iterative method?

An exact solve costs $O(nd^2+d^3)$, one full gradient costs $O(nd)$, and a
minibatch gradient costs $O(|B|d)$. The next cell times these three operations
at $n=20{,}000$ and $d=2{,}000$ to make the per-update trade-off concrete.

Do not read the timing table as an accuracy comparison: the exact solve finishes
the optimization, while a gradient is only one step toward the answer. Its point
is that a very cheap step can be attractive if enough such steps converge.

In [ ]:
def _timed(fn):
    t0 = time.perf_counter(); fn(); return time.perf_counter()-t0

def best_of(fn, reps=3):
    return min(_timed(fn) for _ in range(reps))*1000

n_cost, d_cost = 20_000, 2000
Xc = np.random.default_rng(1).standard_normal((n_cost, d_cost))
yc = np.random.default_rng(2).standard_normal(n_cost)
th = np.zeros(d_cost)

print('exact solve      %8.1f ms' % best_of(lambda: np.linalg.solve(Xc.T@Xc, Xc.T@yc)))
print('full gradient    %8.1f ms' % best_of(lambda: Xc.T@(Xc@th-yc)/n_cost))
print('minibatch (128)  %8.2f ms' % best_of(lambda: Xc[:128].T@(Xc[:128]@th-yc[:128])/128))
del Xc, yc, th

Each row reports the cost of one operation, not
the cost of reaching a common accuracy. The exact solve finishes immediately;
the other two rows produce directions that must be used repeatedly. A minibatch
method is worthwhile when its much cheaper, noisier updates make enough progress
to offset the need for more iterations.

## 2. Why the learning rate matters

The negative gradient tells us which direction decreases the objective, but not
how far to move. We first remove every other complication by using
$f(x)=x^2$. Here one update multiplies $x$ by $1-2\eta$, so the contraction
factor is $|1-2\eta|$.

The next cell runs four learning rates from the same starting point. Compare a
slow monotone path, a faster monotone path, an oscillating but shrinking path,
and an expanding path. The boundary is exact: convergence requires
$0<\eta<1$.

In [ ]:
def descend_1d(eta, steps=10, x0=10.):
    x, path = x0, [x0]
    for _ in range(steps):
        x = x - eta*2*x
        path.append(x)
    return np.array(path)

rates = [.05, .2, .75, 1.1]
labels = ['too small', 'good', 'oscillates', 'diverges']
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
grid = np.linspace(-13, 13, 400)
for ax, eta, label in zip(axes, rates, labels):
    p = descend_1d(eta)
    ax.plot(grid, grid**2, lw=1)
    ax.plot(p, p**2, 'o-', ms=3)
    ax.set(title=f'eta={eta}: {label}', ylim=(-6, 150), xlim=(-13, 13))
plt.tight_layout()

pd.DataFrame({'eta': rates,
              'contraction |1-2eta|': [abs(1-2*e) for e in rates],
              '|x| after 10 steps': [abs(descend_1d(e)[-1]) for e in rates]})

The four paths separate direction from step
length. Every update uses the negative gradient, but $\eta=0.05$ wastes that
direction with tiny moves, $\eta=0.75$ crosses the minimum while contracting,
and $\eta=1.1$ expands. A descent direction alone does not guarantee that the
next iterate has lower loss.

## 3. Gradient descent on least squares

We now move from the toy quadratic to the simulated regression problem. The
first cell implements the least-squares loss, its gradient, and the update loop;
it then verifies that full-batch gradient descent reaches the closed-form
solution.

The following contour plot reruns the same algorithm at $0.08$, $1.0$, and
$2.15$ times $1/\lambda_{\max}$. Compare the length and shape of the paths:
small steps make little progress, stable steps reach the star, and a step beyond
$2/\lambda_{\max}$ moves away from it.

In [ ]:
def loss(theta):
    return .5*np.mean((X@theta-y)**2)

def gradient(theta, indices=None):
    Xb, yb = (X, y) if indices is None else (X[indices], y[indices])
    return Xb.T@(Xb@theta-yb)/len(Xb)

def optimize(kind='gd', eta=.2, steps=100, batch_size=20, beta=.9, start=(-2., 2.4)):
    theta = np.array(start, dtype=float)
    velocity = np.zeros_like(theta)
    history = []
    for step in range(steps):
        g = gradient(theta) if kind == 'gd' else gradient(theta, rng.choice(len(X), batch_size, replace=False))
        velocity = beta*velocity + g if kind == 'momentum' else g
        theta = theta - eta*velocity
        history.append((step, loss(theta), *theta, np.linalg.norm(g)))
    return theta, pd.DataFrame(history, columns=['step','loss','theta1','theta2','grad_norm'])

theta_gd, gd = optimize('gd', eta=1.0, steps=300)
print('gradient descent:', theta_gd.round(4))
print('closed form:     ', theta_star.round(4))
assert np.allclose(theta_gd, theta_star, atol=1e-3)

The matching vectors confirm that the gradient
formula and update converge to the known least-squares solution at a stable
learning rate. That endpoint does not show *how* the algorithm arrived there,
so the next cell exposes the parameter path at three step sizes on the loss
contours.

In [ ]:
lmax = np.linalg.eigvalsh(H).max()
t1, t2 = np.meshgrid(np.linspace(-2.4, 2.8, 160), np.linspace(-2.4, 2.8, 160))
flat = np.column_stack([t1.ravel(), t2.ravel()])
surface = (.5*np.mean((X@flat.T - y[:, None])**2, axis=0)).reshape(t1.shape)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, scale, name in zip(axes, [.08, 1.0, 2.15], ['small','stable','unstable']):
    eta = scale/lmax
    _, path = optimize('gd', eta=eta, steps=20)
    ax.contour(t1, t2, surface, levels=[.2,.35,.6,1.,1.8,3.], colors='0.8', linewidths=.7)
    ax.plot(path.theta1, path.theta2, 'o-', ms=3, lw=1)
    ax.scatter(*theta_star, marker='*', s=90, zorder=5)
    ax.set(title=f'{name}: eta={eta:.2f}', xlabel='theta_1')
axes[0].set_ylabel('theta_2'); plt.tight_layout();

The elongated contours reveal why curvature
matters. One learning rate must be safe in the steep direction, but that same
rate can make progress painfully slow in the flat direction. Increasing the
step helps only until it crosses the stability boundary.

## 4. Minibatch gradients and their noise

Computing a full gradient still reads all $n$ observations at every update. A
minibatch uses only $B$ observations, making an update cheaper but random.

For squared-error regression, observation $i$ contributes

$$\ell_i(\theta)=\frac12(x_i^\top\theta-y_i)^2,
\qquad
g_i(\theta)=(x_i^\top\theta-y_i)x_i.$$

The full gradient is the average of all $n$ example gradients. A minibatch
gradient replaces that average by
$g_B(\theta)=B^{-1}\sum_{i\in B}g_i(\theta)$. Thus SGD is the special case
$B=1$, while full-batch gradient descent uses $B=n$.

The first cell writes out one single-observation update before placing the same
calculation inside a loop. It tracks distance to the exact least-squares
solution so we can see whether the noisy updates actually converge. The final
table reports coefficient errors and the objective gap, rather than only placing
two vectors side by side.

The second cell holds the parameter fixed, repeatedly samples batches, and
checks that the average minibatch gradient approaches the full gradient while
its RMS error falls approximately like $1/\sqrt{B}$. The third cell lets that
noise enter optimization. Compare the objective gaps and parameter paths for
the full batch, $B=32$, and $B=1$.

### Learning-rate decay and tail averaging

With one randomly selected observation $I_t$, the SGD recursion is

$$
\theta_{t+1}
=\theta_t-\eta_t g_{I_t}(\theta_t),
\qquad
g_{I_t}(\theta_t)
=(x_{I_t}^\top\theta_t-y_{I_t})x_{I_t}.
$$

A constant $\eta_t$ keeps every random gradient perturbation equally large.
To let the iterates settle, the code uses a power-law decay

$$
\eta_t=\frac{\eta_0}{t^\gamma},
\qquad \eta_0=0.5,\quad \gamma=0.55.
$$

Early steps are large enough to move toward the optimum; later steps are small
enough to reduce jitter. The choice $1/2<\gamma\le1$ also satisfies the
standard stochastic-approximation conditions
$\sum_t\eta_t=\infty$ and $\sum_t\eta_t^2<\infty$: the algorithm does not
freeze after a finite total movement, while the accumulated squared noise is
controlled.

Even late iterates still fluctuate. After a burn-in of $T_0=30{,}000$ updates,
we report the **tail average**

$$
\bar\theta_T
=\frac{1}{T-T_0}\sum_{t=T_0+1}^{T}\theta_t,
\qquad T=50{,}000.
$$

In the code, `tail_sum` stores the numerator and `tail_count` stores
$T-T_0$. This averaging does not change the SGD recursion; it combines the late
parameter estimates so that fluctuations in opposite directions can cancel.

In [ ]:
# One SGD update, written without a helper function.
example_index = 0
theta_before = np.zeros(X.shape[1])
x_i, y_i = X[example_index], y[example_index]
prediction_i = x_i @ theta_before
residual_i = prediction_i - y_i
one_example_gradient = residual_i * x_i
theta_after = theta_before - 0.2 * one_example_gradient

print(f"example {example_index}: prediction={prediction_i:.3f}, "
      f"response={y_i:.3f}, residual={residual_i:.3f}")
print("one-example gradient:", one_example_gradient.round(4))
print("theta after one update:", theta_after.round(4))

# Now repeat the same update with random rows and a decreasing learning rate.
sgd_rng = np.random.default_rng(SEED + 1)
theta_sgd = np.zeros(X.shape[1])
tail_sum = np.zeros_like(theta_sgd)
tail_count = 0
progress = []
initial_step = 0.5
decay_power = 0.55
burn_in = 30_000
total_steps = 50_000
for t in range(1, total_steps + 1):
    i = sgd_rng.integers(len(X))
    example_gradient = (X[i] @ theta_sgd - y[i]) * X[i]
    learning_rate = initial_step / t**decay_power
    theta_sgd -= learning_rate * example_gradient
    if t > burn_in:
        tail_sum += theta_sgd
        tail_count += 1
    if t % 500 == 0:
        averaged_distance = (np.linalg.norm(tail_sum / tail_count - theta_star)
                             if tail_count else np.nan)
        progress.append((t, np.linalg.norm(theta_sgd - theta_star),
                         averaged_distance))

theta_sgd_averaged = tail_sum / tail_count
sgd_progress = pd.DataFrame(
    progress, columns=["iteration", "current iterate", "tail average"]
).set_index("iteration")
ax = sgd_progress.plot(logy=True, figsize=(5.5, 3.0))
ax.set(ylabel="distance to exact least-squares solution",
       title="decreasing steps and averaging stabilize SGD")

coefficient_comparison = pd.DataFrame({
    "exact least squares": theta_star,
    "last SGD iterate": theta_sgd,
    "tail-averaged SGD": theta_sgd_averaged,
    "last-iterate error": np.abs(theta_sgd - theta_star),
    "averaged error": np.abs(theta_sgd_averaged - theta_star),
}, index=["theta_1", "theta_2"])
display(coefficient_comparison.round(4))
print("last-iterate parameter distance:",
      round(np.linalg.norm(theta_sgd - theta_star), 5))
print("averaged parameter distance:",
      round(np.linalg.norm(theta_sgd_averaged - theta_star), 5))
print("last-iterate objective gap:",
      round(loss(theta_sgd) - loss(theta_star), 8))
print("averaged objective gap:",
      round(loss(theta_sgd_averaged) - loss(theta_star), 8))

Each update uses only one observation, so the
current iterate moves in response to whichever row was sampled most recently.
The decreasing learning rate makes those movements smaller over time.

The tail average is a separate post-processing step; it does not alter any SGD
update. Positive and negative fluctuations partly cancel, so the averaged
parameter can be closer to the optimum than the final random iterate. The
distance plot shows this over the latter part of the run, while the table and
objective gaps quantify the final difference.

### Minibatch averaging reduces gradient noise

At a fixed parameter value, a random minibatch gradient is centered on the full
gradient. Averaging $B$ roughly independent example gradients divides the
gradient variance by $B$, so its typical error decreases like $1/\sqrt{B}$.

The next table estimates this relationship using 500 random batches at each
size:

- `bias_of_mean` is the distance between the average of those 500 estimates and
  the full gradient; it should be close to zero;
- `RMS_error` is the typical distance of one minibatch estimate from the full
  gradient; and
- `sqrt_reference` is an ideal $1/\sqrt{B}$ comparison starting at $B=1$.

The accompanying log--log plot makes the rate easier to see.

In [ ]:
theta_probe = np.array([-.8, 1.2])
full = gradient(theta_probe)
rows = []
for B in [1,2,4,8,16,32,64,128]:
    est = np.array([gradient(theta_probe, rng.choice(len(X), B, replace=False)) for _ in range(500)])
    rows.append((B, np.linalg.norm(est.mean(0)-full), np.sqrt(np.mean(np.sum((est-full)**2, axis=1)))))
noise = pd.DataFrame(rows, columns=['batch_size','bias_of_mean','RMS_error'])
noise['sqrt_reference'] = noise.RMS_error.iloc[0]/np.sqrt(noise.batch_size)
display(noise.round(4))
ax = noise.plot(x='batch_size', y=['RMS_error', 'sqrt_reference'],
                loglog=True, marker='o', figsize=(5.2, 3.0))
ax.set(ylabel='gradient error', title='averaging within a batch reduces noise');

The mean-gradient bias remains small at every
batch size, which is the empirical signature of an unbiased estimate. The RMS
error drops from about $1.25$ at $B=1$ to $0.12$ at $B=128$, close to the
$1/\sqrt{B}$ reference.

This is **gradient averaging**: several example gradients are combined before
one parameter update. It makes each direction more reliable, but computing a
larger batch also costs more. We next propagate this gradient noise through many
updates.

### Why the optimization path remains noisy

At the least-squares optimum, the *average* of all example gradients is zero;
the individual example gradients generally are not. Their cancellation is exact
for the full gradient, partial for a minibatch, and absent for $B=1$. Therefore a
small batch can keep pushing the parameter away from the optimum even after it
has arrived nearby.

The next cell gives all methods the same constant learning rate and the same
number of updates to isolate this effect. The horizontal axis counts updates,
not data points processed: one full-gradient update is much more expensive than
one single-example update.

In [ ]:
# The noisy gradient estimates become noisy optimization paths.
floor = loss(theta_star)
histories = []
for kind, batch_size, label in [
    ('gd', len(X), 'full gradient'),
    ('sgd', 32, 'minibatch, B=32'),
    ('sgd', 1, 'SGD, B=1'),
]:
    _, history = optimize(kind, eta=.35, steps=250, batch_size=batch_size)
    histories.append((label, history))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for label, history in histories:
    axes[0].semilogy(history.step, np.maximum(history.loss-floor, 1e-12), label=label)
    axes[1].plot(history.theta1, history.theta2, alpha=.85, label=label)
axes[0].set(xlabel='update', ylabel='objective gap', title='noise remains near the minimum')
axes[1].scatter(*theta_star, marker='*', s=90, color='black', zorder=5)
axes[1].set(xlabel='theta_1', ylabel='theta_2', title='parameter paths')
for ax in axes: ax.legend()
plt.tight_layout();

Full-batch gradient descent approaches the
minimum smoothly because every update uses the exact average. The $B=32$ path
wanders in a smaller neighborhood, while $B=1$ has the widest fluctuations.
All are directed correctly on average.

With a constant learning rate, gradient noise is multiplied by the same step
size forever, so the objective reaches a noisy plateau rather than converging
exactly. There are two different averaging effects in this section:

| averaging operation | what is averaged | main effect |
|---|---|---|
| minibatch gradient | $B$ example gradients before an update | reduces noise in the update direction |
| tail/iterate average | late parameter vectors after updates | reduces noise in the reported parameter |

Decreasing the learning rate shrinks the plateau; iterate averaging makes the
remaining fluctuations cancel. Jitter with small batches is expected behavior,
not by itself evidence of a bug.

## 5. Momentum for an ill-conditioned objective

On an elongated loss surface, ordinary gradient descent repeatedly reverses in
the steep direction while making slow, consistent progress in the flat one.
Heavy-ball momentum accumulates persistent directions and cancels alternating
ones.

The next cell constructs a quadratic with condition number $100$ and compares
optimally tuned gradient descent with heavy-ball momentum. Inspect both the
parameter paths and the log-scale distance to the minimum. On a quadratic, the
best contraction factor improves from
$(\kappa-1)/(\kappa+1)$ to
$(\sqrt\kappa-1)/(\sqrt\kappa+1)$ per step.

In [ ]:
A = np.diag([1., 100.])
M, m = 100., 1.
kap = M/m
eta_gd = 2/(M+m)
beta_hb = ((np.sqrt(kap)-1)/(np.sqrt(kap)+1))**2
eta_hb = 4/(np.sqrt(M)+np.sqrt(m))**2

def quad_path(eta, beta, steps=40, start=(-3., 1.)):
    p = np.array(start, dtype=float); v = np.zeros(2); out = [p.copy()]
    for _ in range(steps):
        v = beta*v + A@p
        p = p - eta*v
        out.append(p.copy())
    return np.array(out)

gd_path, hb_path = quad_path(eta_gd, 0.), quad_path(eta_hb, beta_hb)
print(f'GD : eta={eta_gd:.4f}                   rate/step={(kap-1)/(kap+1):.4f}')
print(f'HB : eta={eta_hb:.4f}  beta={beta_hb:.3f}   rate/step={(np.sqrt(kap)-1)/(np.sqrt(kap)+1):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, path, name in zip(axes[:2], [gd_path, hb_path], ['gradient descent','heavy ball']):
    ax.plot(path[:,0], path[:,1], 'o-', ms=3, lw=.9)
    ax.scatter([0],[0], marker='*', s=90)
    ax.set(title=name, xlabel='theta_1', ylim=(-4.2, 4.2))
axes[0].set_ylabel('theta_2')
for path, name in [(gd_path,'gradient descent'), (hb_path,'heavy ball')]:
    axes[2].semilogy(np.linalg.norm(path,axis=1)/np.linalg.norm(path[0]), label=name)
axes[2].set(xlabel='iteration', ylabel='relative error', title='distance to the minimum')
axes[2].legend(); plt.tight_layout();

Momentum averages recent gradient directions.
The steep component keeps reversing and largely cancels; the persistent flat
component accumulates. The learning rate and momentum coefficient must be
chosen together because both change the recurrence.

In [ ]:
for b in [.9, .95, .99]:
    sk = (1+np.sqrt(b))/(1-np.sqrt(b))
    print(f'beta={b}: implied kappa={sk**2:,.0f}, memory 1/(1-beta)={1/(1-b):.0f}')

A larger momentum coefficient gives past
gradients a longer effective lifetime: roughly $1/(1-\beta)$ iterations. It is
not automatically better. Long memory can help on a severely elongated
objective, but it also delays the response when the useful direction changes
and can amplify oscillation if the learning rate is not adjusted with it.

## 6. How the update changes for the lasso

Lecture 4 previewed this update when fitting the `Credit` data. Here we isolate
the algorithmic idea and apply it at larger scale. Ordinary gradient descent
cannot be applied at the lasso's nondifferentiable corner. Proximal gradient
keeps the usual gradient step for the squared-error part and follows it with
soft thresholding for the penalty.

The first cell plots the thresholding map and makes its exact-zero region
visible. The second uses that update in a problem with 500 features but only 10
true signals. Compare the number of true signals recovered with the number of
false positives; sparsity alone is not enough if the wrong variables survive.

In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z)*np.maximum(np.abs(z)-gamma, 0.)

grid = np.linspace(-3, 3, 400)
plt.figure(figsize=(4.2, 2.8))
plt.plot(grid, soft_threshold(grid, 1.), lw=1.6)
plt.plot(grid, grid, ls='--', lw=.8, color='0.6')
plt.axhline(0, color='0.85', lw=.8); plt.axvline(0, color='0.85', lw=.8)
plt.title('soft thresholding, gamma=1'); plt.xlabel('z'); plt.ylabel('S(z)');

The dashed line is the identity map, which would
leave a coefficient unchanged. Soft thresholding moves every value toward zero;
all inputs between $-1$ and $1$ become exactly zero in this picture. In a lasso
update the threshold is $\eta\lambda$, so both the learning rate and penalty
determine how much is removed after the gradient step.

The next cell applies this update to a deliberately sparse problem and checks
true discoveries and false positives separately.

In [ ]:
# High-dimensional sparse recovery: d=500 features, only 10 of them real.
gen = np.random.default_rng(7)
n_l, d_l, s_l = 200, 500, 10
Xl = gen.standard_normal((n_l, d_l))
theta_true = np.zeros(d_l)
support = gen.choice(d_l, s_l, replace=False)
theta_true[support] = gen.choice([-1., 1.], s_l)*gen.uniform(1.5, 3., s_l)
yl = Xl@theta_true + .5*gen.standard_normal(n_l)

def proximal_gradient(lmbda, steps=400):
    eta = 1/np.linalg.eigvalsh(Xl.T@Xl/n_l).max()
    th = np.zeros(d_l)
    for _ in range(steps):
        th = soft_threshold(th - eta*(Xl.T@(Xl@th-yl)/n_l), eta*lmbda)
    return th

theta_hat = proximal_gradient(.30)
found, truth = set(np.flatnonzero(theta_hat)), set(support)
print('nonzero coefficients:', len(found), 'of', d_l)
print('true features recovered:', len(truth & found), 'of', s_l)
print('false positives:', len(found - truth))

For this simulated dataset and penalty, proximal
gradient retains all 10 true features and introduces no false positives. That is
a check that the update works in a controlled example, not a general guarantee:
changing the noise, correlations, sample size, or penalty can create both missed
signals and false discoveries.

## Verify the implementation with PyTorch

The final cell checks our from-scratch result against a standard autodiff
implementation. Autodiff changes how the gradient is obtained, not the update
being optimized. With full-batch SGD and the same learning rate, PyTorch should
reach the same least-squares coefficients as the NumPy loop.

In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float64)
y_tensor = torch.tensor(y, dtype=torch.float64).reshape(-1,1)
model = torch.nn.Linear(2, 1, bias=False).double()
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)
for _ in range(300):
    optimizer.zero_grad()
    objective = .5*torch.mean((model(X_tensor)-y_tensor)**2)
    objective.backward()
    optimizer.step()
torch_theta = model.weight.detach().numpy().ravel()
print('PyTorch:     ', torch_theta.round(4))
print('from scratch:', theta_gd.round(4))
assert np.allclose(torch_theta, theta_gd, atol=1e-3)

The matching coefficients confirm that our
NumPy gradient and update convention agree with PyTorch's autodiff and optimizer.
This is an implementation check; it does not by itself establish that the
linear model is appropriate for a particular dataset.

## Try It

For each change, predict the plot or failure mode before running the cell.

1. In the one-dimensional example, try $\eta=0.5$, $1.0$, and $1.01$. What is special about the first two boundary values?
2. Change the simulated condition number from 8 to 80. Which direction slows down, and how must the stable learning-rate range change?
3. Repeat one-example SGD with 20 random seeds. Compare the parameter distance of the last iterate with the tail average. Which has less run-to-run variation?
4. Compare batch sizes at a fixed number of processed observations rather than a fixed number of updates. How does that change the apparent speed comparison?
5. In the sparse-recovery example, sweep the lasso penalty from 0.05 to 1.0. Track false positives and missed true features separately.